# **AMFU-NET PAPER ARCHITECTURE TRAINING**

_Chen et al. AMFU-Net segmentation network with SRAD+CLAHE replacing only the unavailable reflection/transmission fusion input._


In [ ]:
!pip uninstall -y -q transformers
!pip install -q "transformers==4.57.1" "accelerate>=1,<2" "datasets>=3,<5" "huggingface_hub>=0.30,<1" "opencv-python-headless>=4.9,<5" "tensorboard>=2,<3" "scipy>=1.12,<2" "scikit-learn>=1.4,<2"


In [ ]:
from __future__ import annotations

import json
import math
import os
import random

# More stable for repositories containing thousands of PNG files.
os.environ["HF_HUB_DISABLE_XET"] = "1"
# Avoid ConvTranspose2d/DataParallel CUDA misaligned-address failures.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import concatenate_datasets, load_dataset
from huggingface_hub import HfApi, create_repo, login, snapshot_download
from kaggle_secrets import UserSecretsClient
from scipy.ndimage import binary_erosion, distance_transform_edt
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    EarlyStoppingCallback, PretrainedConfig, PreTrainedModel,
    Trainer, TrainingArguments
)
from transformers.trainer_utils import get_last_checkpoint
import transformers

print("Transformers:", transformers.__version__)
assert transformers.__version__ == "4.57.1", "Restart the Kaggle session, then run from the first cell."


@dataclass
class Config:
    seed: int = 42
    data_repo: str = "aee4/G17-preprocessed-dataset"
    model_repo: str = "aee4/G17-AMFU-Net-Paper"
    work_dir: str = "/kaggle/working/G17_amfu_paper_training"
    image_size: int = 256
    batch_size: int = 4
    epochs: int = 200
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    num_workers: int = 2
    patience: int = 200
    base_channels: int = 32
    threshold: float = 0.5
    gradient_accumulation: int = 3
    eval_steps: int = 250
    logging_steps: int = 25
    private_repo: bool = True
    resume: bool = True


CFG = Config()
WORK_DIR = Path(CFG.work_dir)
RESULTS_DIR = WORK_DIR / "results"
PREDICTION_DIR = RESULTS_DIR / "predictions"
for folder in [WORK_DIR, RESULTS_DIR, PREDICTION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("Visible GPUs:", torch.cuda.device_count())
    print("GPU:", torch.cuda.get_device_name(0))
    assert torch.cuda.device_count() == 1, "Restart the session so CUDA_VISIBLE_DEVICES takes effect."
print(json.dumps(asdict(CFG), indent=2))


### Hugging Face login

The token is read securely from Kaggle Secrets.


In [ ]:
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
hf_api = HfApi(token=HF_TOKEN)

create_repo(
    repo_id=CFG.model_repo,
    repo_type="model",
    private=CFG.private_repo,
    exist_ok=True,
    token=HF_TOKEN,
)

hf_data = load_dataset(
    CFG.data_repo,
    token=HF_TOKEN,
)

required_splits = {"train", "validation", "test"}
missing_splits = required_splits - set(hf_data)
if missing_splits:
    raise ValueError(f"Dataset splits missing: {sorted(missing_splits)}")

print(hf_data)
for split_name, split_data in hf_data.items():
    print(split_name, pd.Series(split_data["dataset"]).value_counts().to_dict())


# Recreate the paper's 8:1:1 split while keeping patient groups separate.
all_data = concatenate_datasets([hf_data["train"], hf_data["validation"], hf_data["test"]])
labels = np.array([f"{d}_{c}" for d, c in zip(all_data["dataset"], all_data["class_label"])])
groups = np.asarray(all_data["patient_id"]).astype(str)
indices = np.arange(len(all_data))

outer = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=CFG.seed)
remaining_idx, test_idx = next(outer.split(indices, labels, groups))
inner = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=CFG.seed + 1)
inner_train, inner_val = next(inner.split(
    remaining_idx, labels[remaining_idx], groups[remaining_idx]
))
train_idx = remaining_idx[inner_train]
val_idx = remaining_idx[inner_val]

paper_data = {
    "train": all_data.select(train_idx.tolist()),
    "validation": all_data.select(val_idx.tolist()),
    "test": all_data.select(test_idx.tolist()),
}
print({name: len(part) for name, part in paper_data.items()})


### Dataset

Load the Parquet shards directly with Hugging Face Datasets.


In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, dataset, training: bool):
        self.dataset = dataset
        self.training = training

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        row = self.dataset[index]
        image = np.asarray(row["image"].convert("L"))
        mask = np.asarray(row["mask"].convert("L"))
        image = cv2.resize(image, (CFG.image_size, CFG.image_size), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (CFG.image_size, CFG.image_size), interpolation=cv2.INTER_NEAREST)

        if self.training:
            if random.random() < 0.5:
                image, mask = np.fliplr(image), np.fliplr(mask)
            angle = random.uniform(-15, 15)
            scale = random.uniform(0.9, 1.1)
            tx = random.uniform(-0.1, 0.1) * CFG.image_size
            ty = random.uniform(-0.1, 0.1) * CFG.image_size
            matrix = cv2.getRotationMatrix2D((CFG.image_size/2, CFG.image_size/2), angle, scale)
            matrix[:, 2] += (tx, ty)
            image = cv2.warpAffine(image, matrix, (CFG.image_size, CFG.image_size),
                                   flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
            mask = cv2.warpAffine(mask, matrix, (CFG.image_size, CFG.image_size),
                                  flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)

        image = np.ascontiguousarray(image, dtype=np.float32) / 255.0
        mask = np.ascontiguousarray(mask > 127, dtype=np.float32)
        image = (image - 0.5) / 0.5
        return {
            "pixel_values": torch.from_numpy(image).unsqueeze(0),
            "labels": torch.from_numpy(mask).unsqueeze(0),
        }




train_set = SegmentationDataset(paper_data["train"], training=True)
val_set = SegmentationDataset(paper_data["validation"], training=False)
test_set = SegmentationDataset(paper_data["test"], training=False)
print({"train": len(train_set), "validation": len(val_set), "test": len(test_set)})


In [ ]:
sample = train_set[0]
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(sample["pixel_values"][0] * 0.5 + 0.5, cmap="gray")
axes[0].set_title("Enhanced image")
axes[1].imshow(sample["labels"][0], cmap="gray")
axes[1].set_title("Ground-truth mask")
for axis in axes: axis.axis("off")
plt.tight_layout()


### Model

Paper-faithful AMFU-Net equations: DAG (15-17) on four skip connections and MFEF (18-22) at the bottleneck.


In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DAG(nn.Module):
    """Chen et al. equations (15)-(17)."""
    def __init__(self, channels):
        super().__init__()
        self.branch_average = nn.Conv2d(channels, channels, 3, padding=1)
        self.branch_maximum = nn.Conv2d(channels, channels, 3, padding=1)
        self.output = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        f1 = torch.sigmoid(self.branch_average(x))
        f2 = torch.sigmoid(self.branch_maximum(x))
        g1 = F.adaptive_avg_pool2d(f1, 1)
        g2 = F.adaptive_max_pool2d(f2, 1)
        return self.output(x * g1 + x * g2)


class MFEF(nn.Module):
    """Chen et al. equations (18)-(22)."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 1)
        self.conv3 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv5 = nn.Conv2d(channels, channels, 5, padding=2)
        self.gap_projection = nn.Conv2d(channels, channels, 1)
        self.compress = nn.Conv2d(channels * 4, channels, 1)
        self.dilated = nn.Conv2d(channels, channels, 3, padding=2, dilation=2)

    def forward(self, x):
        size = x.shape[-2:]
        global_feature = self.gap_projection(F.adaptive_avg_pool2d(x, 1))
        global_feature = F.interpolate(global_feature, size=size, mode="bilinear", align_corners=False)
        multi_scale = torch.cat([
            self.conv1(x), self.conv3(x), self.conv5(x), global_feature
        ], dim=1)
        m1 = F.relu(self.compress(multi_scale), inplace=True)
        w1 = torch.softmax(m1, dim=1)
        m2 = F.relu(self.dilated(x), inplace=True)
        w2 = torch.softmax(m2, dim=1)
        weights = 0.5 * w1 + 0.5 * w2
        return x * weights


class AMFUConfig(PretrainedConfig):
    model_type = "amfu-net"
    def __init__(self, base_channels=32, image_size=256, **kwargs):
        super().__init__(**kwargs)
        self.base_channels = base_channels
        self.image_size = image_size


class AMFUNet(PreTrainedModel):
    config_class = AMFUConfig

    def __init__(self, config):
        super().__init__(config)
        b = config.base_channels
        self.enc1 = ConvBNReLU(1, b)
        self.enc2 = ConvBNReLU(b, b * 2)
        self.enc3 = ConvBNReLU(b * 2, b * 4)
        self.enc4 = ConvBNReLU(b * 4, b * 8)
        self.bottleneck = ConvBNReLU(b * 8, b * 16)
        self.pool = nn.MaxPool2d(2)
        self.mfef = MFEF(b * 16)

        self.dag1, self.dag2 = DAG(b), DAG(b * 2)
        self.dag3, self.dag4 = DAG(b * 4), DAG(b * 8)
        self.up4 = nn.ConvTranspose2d(b * 16, b * 8, 2, 2)
        self.dec4 = ConvBNReLU(b * 16, b * 8)
        self.up3 = nn.ConvTranspose2d(b * 8, b * 4, 2, 2)
        self.dec3 = ConvBNReLU(b * 8, b * 4)
        self.up2 = nn.ConvTranspose2d(b * 4, b * 2, 2, 2)
        self.dec2 = ConvBNReLU(b * 4, b * 2)
        self.up1 = nn.ConvTranspose2d(b * 2, b, 2, 2)
        self.dec1 = ConvBNReLU(b * 2, b)
        self.head = nn.Conv2d(b, 1, 1)
        self.post_init()

    def forward(self, pixel_values, labels=None):
        s1 = self.enc1(pixel_values)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))
        x = self.mfef(self.bottleneck(self.pool(s4)))
        x = self.up4(x); x = self.dec4(torch.cat([x, self.dag4(s4)], dim=1))
        x = self.up3(x); x = self.dec3(torch.cat([x, self.dag3(s3)], dim=1))
        x = self.up2(x); x = self.dec2(torch.cat([x, self.dag2(s2)], dim=1))
        x = self.up1(x); x = self.dec1(torch.cat([x, self.dag1(s1)], dim=1))
        logits = self.head(x)
        loss = None
        if labels is not None:
            probs = torch.sigmoid(logits)
            dims = (1, 2, 3)
            intersection = (probs * labels).sum(dims)
            dice = (2 * intersection + 1.0) / (probs.sum(dims) + labels.sum(dims) + 1.0)
            loss = F.binary_cross_entropy_with_logits(logits, labels) + (1 - dice.mean())
        return {"loss": loss, "logits": logits}


model_config = AMFUConfig(base_channels=CFG.base_channels, image_size=CFG.image_size)
model = AMFUNet(model_config)
parameters = sum(p.numel() for p in model.parameters())
with torch.inference_mode():
    output = model(torch.zeros(2, 1, CFG.image_size, CFG.image_size))["logits"]
assert output.shape == (2, 1, CFG.image_size, CFG.image_size)
print(f"Parameters: {parameters:,}")


### Evaluation metrics


In [ ]:
def surface_distances(prediction, target):
    prediction = prediction.astype(bool)
    target = target.astype(bool)
    if not prediction.any() or not target.any():
        return np.nan, np.nan
    pred_surface = prediction ^ binary_erosion(prediction)
    target_surface = target ^ binary_erosion(target)
    target_distance = distance_transform_edt(~target_surface)
    pred_distance = distance_transform_edt(~pred_surface)
    distances = np.concatenate([target_distance[pred_surface], pred_distance[target_surface]])
    return float(np.percentile(distances, 95)), float(np.mean(distances))


def compute_metrics(prediction):
    logits = prediction.predictions
    labels = prediction.label_ids >= 0.5
    predictions = 1 / (1 + np.exp(-np.clip(logits, -30, 30))) >= CFG.threshold
    dice, jaccard, hd95, asd = [], [], [], []
    for pred, target in zip(predictions[:, 0], labels[:, 0]):
        tp = np.logical_and(pred, target).sum()
        fp = np.logical_and(pred, ~target).sum()
        fn = np.logical_and(~pred, target).sum()
        eps = 1e-7
        dice.append((2*tp+eps)/(2*tp+fp+fn+eps))
        jaccard.append((tp+eps)/(tp+fp+fn+eps))
        h, a = surface_distances(pred, target)
        hd95.append(h); asd.append(a)
    return {
        "dice": float(np.mean(dice)),
        "jaccard": float(np.mean(jaccard)),
        "hd95": float(np.nanmean(hd95)),
        "asd": float(np.nanmean(asd)),
    }


### Cross-session resume setup

Pull `last-checkpoint` from the model repository when it exists.


In [ ]:
OUTPUT_DIR = str(WORK_DIR / "trainer_output")
LOG_DIR = str(WORK_DIR / "logs")
resume_checkpoint = None

try:
    repo_files = hf_api.list_repo_files(CFG.model_repo, repo_type="model")
    has_checkpoint = any(path.startswith("last-checkpoint/") for path in repo_files)
except Exception:
    has_checkpoint = False

if has_checkpoint and CFG.resume:
    downloaded = snapshot_download(
        repo_id=CFG.model_repo,
        repo_type="model",
        allow_patterns="last-checkpoint/*",
        token=HF_TOKEN,
    )
    resume_checkpoint = str(Path(downloaded) / "last-checkpoint")
    print("Resuming from:", resume_checkpoint)
else:
    print("No Hub checkpoint found; starting fresh")


### Training arguments

Paper settings: Adam, learning rate 0.001, weight decay 1e-5, 200 epochs and effective batch size 12. The T4-safe micro-batch is 4 with three-step accumulation. The paper does not state its loss function, so BCE+Dice is used explicitly.


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=CFG.batch_size,
    per_device_eval_batch_size=CFG.batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation,
    num_train_epochs=CFG.epochs,
    learning_rate=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=CFG.eval_steps,
    save_steps=CFG.eval_steps,
    logging_steps=CFG.logging_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="dice",
    greater_is_better=True,
    fp16=False,
    optim="adamw_torch",
    dataloader_num_workers=CFG.num_workers,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="tensorboard",
    logging_dir=LOG_DIR,
    push_to_hub=True,
    hub_model_id=CFG.model_repo,
    hub_strategy="checkpoint",
    hub_private_repo=CFG.private_repo,
    remove_unused_columns=False,
)


In [ ]:
class PaperTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            self.optimizer = torch.optim.Adam(
                self.model.parameters(),
                lr=self.args.learning_rate,
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer

    def create_scheduler(self, num_training_steps, optimizer=None):
        if self.lr_scheduler is None:
            steps_per_epoch = math.ceil(
                len(self.train_dataset) /
                (self.args.per_device_train_batch_size * self.args.gradient_accumulation_steps)
            )
            self.lr_scheduler = torch.optim.lr_scheduler.StepLR(
                self.optimizer,
                step_size=steps_per_epoch * 10,
                gamma=0.5,
            )
        return self.lr_scheduler


trainer = PaperTrainer(
    model=model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=val_set,
    compute_metrics=compute_metrics,
)


### Live TensorBoard (optional)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR


In [ ]:
trainer.train(resume_from_checkpoint=resume_checkpoint)


### Training curves

Read the continuous history saved by the Trainer.


In [ ]:
final_checkpoint = get_last_checkpoint(OUTPUT_DIR)
state_path = Path(final_checkpoint) / "trainer_state.json"
state = json.loads(state_path.read_text())
logs = state["log_history"]

def points(key):
    return [(row["step"], row[key]) for row in logs if key in row]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, key, title in zip(
    axes.flat,
    ["loss", "eval_loss", "eval_dice", "eval_jaccard"],
    ["Training loss", "Validation loss", "Validation Dice", "Validation Jaccard"],
):
    values = points(key)
    if values: axis.plot(*zip(*values))
    axis.set_title(title); axis.set_xlabel("Step"); axis.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=160)
plt.show()


### Internal-test evaluation


In [ ]:
test_output = trainer.predict(test_set, metric_key_prefix="test")
print({key: round(value, 4) for key, value in test_output.metrics.items() if isinstance(value, float)})

probabilities = 1 / (1 + np.exp(-np.clip(test_output.predictions, -30, 30)))
predictions = probabilities >= CFG.threshold
test_metadata = paper_data["test"]
rows = []
for index in range(len(test_metadata)):
    row = test_metadata[index]
    label = test_output.label_ids[index] >= 0.5
    pred = predictions[index]
    tp = float(np.logical_and(pred, label).sum())
    fp = float(np.logical_and(pred, ~label).sum())
    fn = float(np.logical_and(~pred, label).sum())
    eps = 1e-7
    rows.append({
        "image_id": row["image_id"], "dataset": row["dataset"],
        "dice": (2*tp+eps)/(2*tp+fp+fn+eps),
        "jaccard": (tp+eps)/(tp+fp+fn+eps),
        "hd95": surface_distances(pred[0], label[0])[0],
        "asd": surface_distances(pred[0], label[0])[1],
    })
    folder = PREDICTION_DIR / row["dataset"]
    folder.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(folder / f"{row['image_id']}.png"), pred[0].astype(np.uint8)*255)

test_results = pd.DataFrame(rows)
test_results.to_csv(RESULTS_DIR / "internal_test_per_image.csv", index=False)
display(test_results.groupby("dataset")[["dice", "jaccard", "hd95", "asd"]].agg(["mean", "std", "count"]))


In [ ]:
indices = np.linspace(0, len(test_set)-1, min(4, len(test_set)), dtype=int)
fig, axes = plt.subplots(len(indices), 3, figsize=(9, 3*len(indices)))
axes = np.atleast_2d(axes)
for r, i in enumerate(indices):
    item = test_set[i]
    axes[r,0].imshow(item["pixel_values"][0]*0.5+0.5, cmap="gray")
    axes[r,1].imshow(item["labels"][0], cmap="gray")
    axes[r,2].imshow(predictions[i,0], cmap="gray")
    for axis in axes[r]: axis.axis("off")
axes[0,0].set_title("Enhanced"); axes[0,1].set_title("Ground truth"); axes[0,2].set_title("Prediction")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "prediction_examples.png", dpi=160)
plt.show()


### Save final model to Hugging Face


In [ ]:
trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub(commit_message="Complete G17 AMFU-Net training")
hf_api.upload_folder(
    folder_path=str(RESULTS_DIR), path_in_repo="results",
    repo_id=CFG.model_repo, repo_type="model",
)
print(f"Saved to https://huggingface.co/{CFG.model_repo}")
